# Gemini Notebook (formerly **NotebookLM**) — Graduate Study Notes

> **Concise, verifiable notes on Google's source-grounded research assistant.**
> Written for someone who needs to *understand the system*, not just click the buttons.

---

**Status of this document:** current as of **2026-07-26**.
Google renamed **NotebookLM → Gemini Notebook** on **2026-07-16**. Both names appear in the wild; the URL `notebooklm.google.com` still resolves. These notes use **"Notebook"** generically and flag the rebrand where it matters.

**How to read these notes**
| Section | Purpose |
|---|---|
| §1–§3 | Conceptual model — what class of system this is |
| §4–§7 | Mechanics — sources, chat, Studio artifacts |
| §8–§10 | Operating envelope — tiers, limits, privacy |
| §11–§13 | Critical evaluation — failure modes, verification, when *not* to use it |
| §14–§16 | Applied — workflows, prompt patterns, a from-scratch RAG toy |
| §17–§18 | Glossary + self-check |

Code cells are **pure-Python, zero-dependency** (pandas used opportunistically with a fallback). Run top-to-bottom.

---

## 0. TL;DR — the one-paragraph version

Gemini Notebook is a **closed-domain RAG system with a consumer UI**. You upload a bounded corpus (PDFs, Docs, YouTube transcripts, URLs, audio); every generated token is retrieved from and cited back to *that corpus only*. The model underneath is Gemini; the product's value is not the model but the **grounding contract** — refusal to answer from parametric memory, inline citations to exact passages, and a "Studio" layer that renders the same grounded content into podcasts, videos, mind maps, flashcards, quizzes, slide decks, and reports. Its measured hallucination rate is materially lower than open-web chatbots on the same sources (~13% vs ~40% in one study), and its dominant failure mode shifts from *fabrication* to **retrieval miss and misreading** — which is a different, and for a researcher a more tractable, error class.

**The one thing to remember:** *it cannot tell you anything your sources don't say — and that is the feature.*

---
## 1. Lineage & positioning

| Date | Event |
|---|---|
| 2023 | Launches as **Project Tailwind** (Google Labs experiment) |
| 2023–24 | Renamed **NotebookLM**; Gemini 1.5 Pro backend; long-context ingestion |
| Sep 2024 | **Audio Overviews** (two-host podcast) — the feature that made it go viral |
| Dec 2024 | **NotebookLM Enterprise** via Google Cloud; Plus tier |
| 2025 | Mind Maps, **Video Overviews**, Studio expansion, public sharing, mobile apps |
| **2026-07-16** | Rebranded **Gemini Notebook**; per-notebook **secure cloud computer** (native code execution); sync with the Gemini app; Search AI Mode integration announced |
| Now | ~**30M users**, **600k+ organizations** (Google's own figures, Jul 2026) |

### Where it sits in the design space

```
                    grounding source
                 ┌──────────────────────────────────────────┐
                 │ parametric (model weights)               │  ← plain chatbot
                 │ open web (live retrieval)                │  ← Gemini/ChatGPT w/ search
                 │ YOUR bounded corpus  ★ Gemini Notebook   │  ← closed RAG
                 │ your corpus, weights updated             │  ← fine-tuning
                 └──────────────────────────────────────────┘
```

**Key discriminator:** a chatbot with file upload *may* use your file. Notebook *may only* use your files. That constraint is enforced product-side and is what makes citations trustworthy.

---
## 2. Mental model: source grounding

**Retrieval-Augmented Generation (RAG)**, standard form:

$$
\hat{y} \;=\; \operatorname*{arg\,max}_{y} \; p_\theta\!\left(y \mid q,\; \mathcal{R}(q, \mathcal{D})\right)
$$

- $q$ — your question
- $\mathcal{D}$ — the corpus you uploaded (the *sources*)
- $\mathcal{R}$ — the retriever: returns the top-$k$ passages relevant to $q$
- $p_\theta$ — Gemini, generating **conditioned on those passages**

Notebook adds two product-level constraints that plain RAG does not guarantee:

1. **Closed world.** $\theta$'s parametric knowledge is suppressed as an *answer source*. Ask it something outside $\mathcal{D}$ and the correct behavior is a refusal ("the sources do not discuss…"), not a guess.
2. **Attribution.** Every claim carries a citation chip resolving to the exact passage in the exact source. Attribution is *generated alongside* the answer, not reconstructed afterward.

### Long context vs. chunked retrieval

Gemini's very large context window (order $10^6$ tokens) means Notebook does **not** have to chunk aggressively for small/medium corpora — it can feed whole documents and preserve document structure. For large corpora it falls back to selective retrieval. Practical consequence:

> **Small, curated corpora behave qualitatively better than large sprawling ones.** With 8 well-chosen papers the model effectively "sees everything." With 250 sources you are back to hoping the retriever picked the right passages. *Curation is the highest-leverage thing you control.*

In [ ]:
# --- Budget estimator: will my corpus fit "in view", or will it be retrieved selectively? ---
# Rule of thumb for English prose: 1 word ~ 1.33 tokens (Gemini/GPT-family BPE tokenizers).

WORDS_PER_TOKEN = 1 / 1.33
PAGE_WORDS      = 500          # dense academic page
CONTEXT_TOKENS  = 1_000_000    # order-of-magnitude working context

def estimate(n_sources, pages_per_source, page_words=PAGE_WORDS):
    words  = n_sources * pages_per_source * page_words
    tokens = words * 1.33
    return words, tokens, tokens / CONTEXT_TOKENS

scenarios = [
    ("Seminar week (5 papers x 20p)",        5,  20),
    ("Lit review (40 papers x 20p)",        40,  20),
    ("Full course (12 lectures x 40p)",     12,  40),
    ("Thesis corpus (250 papers x 20p)",   250,  20),
    ("One 500-page book",                    1, 500),
]

print(f"{'scenario':<34}{'words':>10}{'tokens':>12}{'frac of 1M ctx':>16}")
print("-" * 72)
for name, n, p in scenarios:
    w, t, frac = estimate(n, p)
    flag = "  <- fits whole" if frac <= 1 else "  <- retrieval-limited"
    print(f"{name:<34}{w:>10,.0f}{t:>12,.0f}{frac:>15.2f}x{flag}")

print("\nPer-source hard cap: 500,000 words OR 200 MB per uploaded file.")
print("=> A single source is capped well below the context window; the *notebook* is what overflows.")

---
## 3. Anatomy of the interface

Three panes. Learn them as three distinct **roles**, not three tabs.

```
┌───────────────┬────────────────────────────┬───────────────────┐
│  SOURCES      │          CHAT              │      STUDIO       │
│  (the corpus) │   (grounded Q&A)           │  (artifacts)      │
├───────────────┼────────────────────────────┼───────────────────┤
│ • add/remove  │ • ask, get cited answers   │ • Audio Overview  │
│ • ☑ select    │ • follow-ups keep context  │ • Video Overview  │
│   subset      │ • "Save to note"           │ • Mind Map        │
│ • per-source  │ • suggested questions      │ • Reports         │
│   auto-summary│ • refuses off-corpus Qs    │ • Flashcards/Quiz │
│ • Discover    │                            │ • Slide/Infogfx   │
│   sources     │                            │ • Data Table      │
└───────────────┴────────────────────────────┴───────────────────┘
                          ▲
              the ☑ checkboxes are a QUERY SCOPE control —
              the most underused feature in the product
```

### The scoping trick (do this)

Deselect everything except the 2–3 sources you actually want reasoned over. You have just converted a noisy 40-source retrieval problem into a near-deterministic long-context read. Use it for:
- comparing exactly two papers,
- quoting one primary text without contamination from commentary,
- isolating a single lecture before an exam.

---
## 4. Sources — what goes in

### Accepted types (as of Jul 2026)

| Class | Formats / notes |
|---|---|
| Documents | `.pdf`, `.txt`, `.md`, `.docx`, `.pptx`, `.csv`, ePub |
| Google Drive | Docs, Slides (≤100), Sheets (≤100k tokens) — **auto-syncs** every few minutes |
| Web | URLs (public, non-paywalled) |
| Video | **YouTube** — public, **captioned**, uploaded >72h ago |
| Audio | `.mp3`, `.wav`, … (transcribed on ingest) |
| Images | png, jpg, webp, heic, tif, gif, bmp, avif, … |
| Other | Pasted text; Gemini chat transcripts as context |

### Limits

| Scope | Limit |
|---|---|
| Per source | **500,000 words** or **200 MB** (no page limit) |
| Sources / notebook | **50** free · 100 Plus · 300 Pro · 500–600 Ultra |
| Notebooks | 100 free · 500 Pro/Ultra |

### Ingestion gotchas — these cause most "it didn't find it" complaints

1. **Scanned PDFs.** No text layer ⇒ nothing to retrieve. OCR first (`ocrmypdf`, Acrobat) or the source is effectively empty.
2. **Paywalled / JS-rendered URLs.** Often ingest as a cookie banner. Verify by opening the source and reading its auto-summary — if the summary is about "accepting cookies," delete and re-upload as PDF.
3. **YouTube without captions.** Silently unusable. Auto-generated captions count.
4. **URL is a snapshot, not a subscription.** Re-add to refresh. (Drive files are the exception — they auto-sync.)
5. **Tables in PDFs** linearize badly. If numbers matter, upload the CSV too.
6. **Sheets are token-capped**, not row-capped — wide sheets truncate earlier than you expect.

> **Verification habit:** after every upload, read the per-source auto-summary. It is a free, one-click integrity check on whether ingestion actually worked.

### Discover / Fast Research / Deep Research
Notebook can *find* sources as well as consume them: a quick web/Drive search that proposes sources for one-click import, and an agentic **Deep Research** mode that browses hundreds of pages and returns a cited report you can then save *as a source*. Treat Deep-Research output as a **secondary source with unvetted provenance** — useful as a map of a literature, never as a citation in your own writing.

---
## 5. Chat — grounded Q&A

**What it is good at**
- *Locative* questions: "Where do these authors define 'alignment tax'?" → citations = a search index over your corpus.
- *Synthetic* questions across sources: "Which papers disagree with Smith's estimate, and on what grounds?"
- *Negative* questions: **"What do my sources NOT address about X?"** — the single most valuable question type for a literature review, and one an open-web chatbot answers badly because it will happily fill the gap.
- Structured extraction: "Build a table: paper | dataset | metric | result | limitation."

**What it is bad at**
- Arithmetic/aggregation over many sources (improving with the 2026 code-execution backend, but verify).
- Anything requiring knowledge you did not upload — by design.
- Judging source *quality*. It grounds in your corpus; it does not audit it. **Garbage in, cited garbage out.**

**Mechanics worth knowing**
- Follow-ups retain conversational context; the *source selection* still gates retrieval.
- **Save to note** pins an answer into the notebook — and a saved note can be **converted into a source**, so your own synthesis becomes retrievable. This is the loop that turns a notebook into a growing knowledge base.
- Chat history is not the permanent artifact; notes and Studio outputs are.

---
## 6. Studio — one corpus, many renderings

The intellectual point of Studio: **the corpus is the asset; every artifact is a projection of it.** Choose the projection that matches the cognitive task.

| Artifact | What it produces | Best cognitive use |
|---|---|---|
| **Audio Overview** | 2-host podcast; **Interactive mode** lets you interrupt and ask; 80+ languages, full-length | Passive review; commute; first pass on an unfamiliar field |
| **Video Overview** | Narrated visuals; formats *explainer* / *brief*; styles (whiteboard, watercolor, kawaii, classic) | Explaining to others; visual learners |
| **Cinematic Video Overview** | Veo-generated video — **Ultra only** | Presentation polish |
| **Mind Map** | Interactive concept hierarchy; click a node to chat about it | Building structure of an unfamiliar domain |
| **Reports** | Study guide, briefing doc, FAQ, timeline, custom | Written deliverables, exam prep |
| **Flashcards** | Q/A pairs, difficulty-tunable | Spaced repetition (export → Anki) |
| **Quiz** | Graded MCQ/short answer w/ explanations | **Testing effect** — the highest-yield study use |
| **Infographic** | Single-image visual summary | Posters, slides, tweet-length dissemination |
| **Slide Deck** | Structured deck | Seminar/journal-club prep |
| **Data Table** | Structured extraction across sources | Systematic review evidence tables ★ |

**Exports:** 12+ formats — PDF, DOCX, PPTX, XLSX, CSV, JSON, Markdown, images. Most artifacts accept a **customization prompt** ("focus on methodology," "assume graduate-level statistics," "critique rather than summarize").

### Pedagogical note (why this matters academically)
Audio/Video Overviews are **recognition**-mode learning: pleasant, low-effort, and weakly correlated with retention. Quizzes and Flashcards are **retrieval-practice** mode: effortful, and the mechanism with the strongest evidence base in the learning-science literature. If your study time is finite, spend it on the right-hand column, not the podcast.

---
## 7. New in 2026 — what the rebrand actually changed

1. **A secure cloud computer per notebook.** Notebook can now *write and execute code* against your sources — real aggregation, statistics, and plots grounded in uploaded data rather than an LLM guessing at arithmetic. Rolled out to Ultra first, then Pro. This is the largest functional change since Audio Overviews: it moves the product from "retrieve + summarize" to "retrieve + compute."
2. **Gemini ecosystem sync.** Notebooks appear in the Gemini app; Search *AI Mode* integration announced.
3. **Drive auto-sync + multi-format export** for Pro/Ultra.
4. **Name.** `NotebookLM` → `Gemini Notebook`. Existing notebooks and links preserved.

> **Exam-style question:** *why does code execution matter more than a bigger context window?* Because grounding fixes **provenance**, not **computation**. A cited-but-wrong sum is still wrong. Executing code moves numerical claims from "sampled from a language model" to "computed deterministically," which is a different epistemic category.

In [1]:
# --- §8 Tiers & limits (as of 2026-07; Google revises these frequently -- verify before quoting) ---
rows = [
    # plan,      USD/mo,  notebooks, src/nb, chats/day, audio/day, deep research/day
    ("Free",        0.00,   100,   50,    50,    3,   "-"),
    ("Plus",        7.99,   500,  100,   "raised", "raised", "limited"),
    ("Pro",        19.99,   500,  300,   500,  "raised",  20),
    ("Ultra 20TB", 99.99,   500,  500,  2500,  100,       75),
    ("Ultra 30TB",199.99,   500,  600,  5000,  100,       75),
]
cols = ["plan","usd_per_mo","notebooks","sources_per_nb","chats_per_day","audio_per_day","deep_research_per_day"]

try:
    import pandas as pd
    df = pd.DataFrame(rows, columns=cols)
    display(df)
except ImportError:
    print(" | ".join(cols))
    for r in rows:
        print(" | ".join(str(x) for x in r))

print("""
Notes:
 * Plus/Pro/Ultra are bundled into Google AI Plus / Pro / Ultra subscriptions --
   you rarely buy "Notebook" alone.
 * Students frequently get Pro free via university offers -- CHECK BEFORE PAYING.
 * Enterprise (Google Cloud) and Workspace editions are priced separately and
   are governed by different data terms (see section 9).
 * Figures compiled from Google support pages + third-party trackers, 2026-07-26.
   Treat as order-of-magnitude, not contractual.""")

,plan,usd_per_mo,notebooks,sources_per_nb,chats_per_day,audio_per_day,deep_research_per_day
0,Free,0.00,100,50,50,3,-
1,Plus,7.99,500,100,raised,raised,limited
2,Pro,19.99,500,300,500,raised,20
3,Ultra 20TB,99.99,500,500,2500,100,75
4,Ultra 30TB,199.99,500,600,5000,100,75



Notes:
 * Plus/Pro/Ultra are bundled into Google AI Plus / Pro / Ultra subscriptions --
   you rarely buy "Notebook" alone.
 * Students frequently get Pro free via university offers -- CHECK BEFORE PAYING.
 * Enterprise (Google Cloud) and Workspace editions are priced separately and
   are governed by different data terms (see section 9).
 * Figures compiled from Google support pages + third-party trackers, 2026-07-26.
   Treat as order-of-magnitude, not contractual.


---
## 9. Privacy, governance, and what you may put in

Three regimes. **Know which one you are in before uploading anything sensitive.**

| Regime | Who | Data used for training? | Human review? | Sharing |
|---|---|---|---|---|
| **Consumer** (personal Google acct) | free/Plus/Pro/Ultra | **No** (Google's stated policy) | Possible on submitted *feedback* | Private by default; optional **public link** |
| **Workspace** (school/work acct) | Workspace core service | **No** | **No**, even on feedback | Within-domain sharing |
| **Enterprise** (Google Cloud) | large orgs | **No** | No | IAM-controlled; **no public sharing**; VPC-SC, audit logs |

**Practical rules for a graduate student**

1. **Use your university Workspace account** for anything involving unpublished work, human-subjects data, or a supervisor's drafts. The data terms are strictly stronger than the consumer tier and it costs you nothing.
2. **IRB/HIPAA/FERPA data:** do not upload without checking your institution's approved-tools list. "Google says it won't train on it" is not the same as "your IRB approved this processor."
3. **Copyright:** uploading a licensed PDF for personal study is generally within normal academic use; generating a *public* Audio Overview from a paywalled book and posting it is redistribution. The tool will not stop you; your publisher might.
4. **Public notebooks** share sources *and* generated content with anyone holding the link. Audit source selection before flipping that switch.
5. **Confidentiality is not the same as accuracy.** A private notebook can still mislead you.

---
## 10. Limitations & failure modes — the section that matters

**Empirically:** in one comparison, Notebook fabricated in ~**13%** of answers vs ~**40%** for general chatbots on the same sources. Better — *not solved*. Grounding changes the **distribution** of errors, not their existence.

### Taxonomy of failure

| # | Failure | Mechanism | Detection |
|---|---|---|---|
| F1 | **Retrieval miss** | Passage exists but wasn't retrieved → confident "the sources don't discuss this" | Re-ask with the author's own vocabulary; deselect other sources |
| F2 | **Misreading** | Retrieved correctly, interpreted wrongly (negation, hedges, limitations sections) | **Click the citation.** Always. |
| F3 | **False synthesis** | Merges two sources' claims into one neither made | Ask "which source states this, verbatim?" |
| F4 | **Provenance laundering** | A weak source is cited with the same visual authority as a strong one | Curate the corpus; Notebook does not rank credibility |
| F5 | **Ingestion silence** | Scanned PDF / caption-less video ingested as near-empty | Read the per-source auto-summary |
| F6 | **Arithmetic drift** | Counts/sums generated rather than computed | Use code execution; verify by hand |
| F7 | **Sycophantic framing** | Studio outputs default to enthusiastic exposition, flattening real disagreement | Prompt explicitly for critique/contradiction |
| F8 | **Corpus siloing** | No cross-notebook querying; knowledge fragments across notebooks | Deliberate notebook architecture (see §14) |

### Structural limitations
- **No true reasoning over the whole corpus at scale** — retrieval is still a bottleneck past ~100 sources.
- **No live web** unless you add sources; a notebook is a **snapshot**, and a stale one silently.
- **Source cap** forces curation decisions on large literatures.
- **Not a citation manager** — no BibTeX, no dedup, no PDF library management. Zotero remains upstream.
- **Non-determinism** — the same question can yield differently-worded answers; artifacts are not reproducible builds.

> **The meta-risk: fluency-induced complacency.** A podcast about papers you have not read *feels* like having read them. The confidence transfers; the comprehension does not. Use overviews to decide **what to read**, never as a substitute for reading what you will cite.

---
## 11. Verification protocol

Treat any Notebook output as a **claim by a research assistant you supervise**.

```
For each claim you intend to rely on:
 1. CLICK the citation chip.
 2. Read the cited passage IN FULL, not the highlighted fragment.
 3. Ask: does the passage assert the claim, or merely mention the topic?
 4. Check scope words: "some", "may", "in mice", "under assumption A", "n = 12".
 5. If the claim aggregates (counts, "most authors", "consistently"),
    re-derive it yourself -- aggregation is F3/F6 territory.
 6. If you will cite it in your own work, open the ORIGINAL source and
    cite that. Never cite the notebook.
```

**Adversarial prompts** (run these before trusting a synthesis):

- "What evidence in the sources *contradicts* the previous answer?"
- "Quote verbatim the sentence that supports this claim."
- "Which of my sources is weakest methodologically, and why?"
- "What would have to be true for this conclusion to be wrong?"
- "List claims you made that are not directly supported by a single source."

---
## 12. When *not* to use it

| Situation | Use instead |
|---|---|
| You need current facts beyond your corpus | Gemini / ChatGPT with live search |
| You need to write code | Claude Code, Cursor, an IDE agent |
| You need reproducible, auditable pipelines | Your own RAG (§16), LangChain/LlamaIndex |
| Sensitive data, non-approved processor | Local LLM (Ollama + a local RAG) |
| Managing 2,000 PDFs and citations | Zotero / Paperpile (feed *subsets* into Notebook) |
| You must genuinely learn the material | Read the paper. Then quiz yourself with Notebook. |
| Numerical analysis you must defend | Jupyter + pandas — this notebook, in fact |

**Complementary stack that actually works:**

```
Zotero  ──(export selected PDFs)──▶  Gemini Notebook  ──(quiz/table/report)──▶  Obsidian
  ▲                                       │                                       │
  └──────── read the originals ◀──────────┘  (triage: what deserves a full read)  │
                                                                                  ▼
                                                                          your own writing
```

---
## 13. Graduate workflows (concrete)

### A. Literature triage — 40 papers, one afternoon
1. Zotero → export 40 PDFs → one notebook, `Topic-Lit-2026`.
2. Studio → **Data Table**: `paper | year | question | method | n | key finding | limitation`. Export XLSX.
3. Chat: *"Group these papers into methodological schools; name the disagreement between them."*
4. Chat: *"Which papers are cited by others in this set as foundational?"*
5. **Mark 6 papers for a real read.** Everything else stays triage-level.
6. Chat: *"What questions does this literature NOT address?"* → candidate contributions.

### B. Seminar / journal club
Notebook = the paper + its 3 key references + your own annotated notes. Generate a **Slide Deck**, then an **Audio Overview** in *critique* framing. Walk in with the discussant's questions pre-generated, then discard the ones you can't defend from the text.

### C. Exam prep (highest evidential yield)
Corpus = lecture slides + notes + textbook chapters. Then **Quiz → grade yourself → re-read only what you missed → Flashcards for those → export to Anki.** Do *not* start with the podcast.

### D. Writing
Separate notebook per chapter. Sources = your literature *plus your own drafts*. Ask: *"Where does my draft assert something my sources do not support?"* — an unusually effective self-critique loop, because the corpus contains both the claim and the evidence.

### E. Methods / code
Upload documentation, a paper's appendix, and your data CSV. With code execution enabled, ask it to reproduce a reported statistic from the raw data — a fast integrity check on published numbers.

### Notebook architecture (because there is no cross-notebook search)
```
one notebook per COHERENT QUESTION, not per topic and not per semester
  ✔  "Chapter 3 — measurement error in survey panels"     (12 sources, sharp)
  ✘  "PhD"                                                (300 sources, useless)
```

In [2]:
# --- §14 Prompt pattern library. Run this cell to print; adapt the {slots}. ---
PATTERNS = {
 "Evidence table": (
    "Build a markdown table over ALL selected sources with columns: "
    "source | year | research question | method | sample | key finding | stated limitation. "
    "One row per source. Write 'not stated' where the source is silent -- do not infer."),
 "Disagreement map": (
    "Identify every point on which my sources disagree about {topic}. For each: state the "
    "positions, name the sources holding each, and quote the sentence that establishes the position."),
 "Gap finding": (
    "What questions about {topic} do these sources raise but NOT answer? Distinguish (a) explicitly "
    "flagged as future work from (b) gaps you infer. Label each."),
 "Steelman / critique": (
    "Argue AGAINST the main claim of {source}, using only evidence from the other selected sources. "
    "If the other sources contain no such evidence, say so explicitly."),
 "Methods extraction": (
    "Describe the methodology of {source} in enough detail that I could replicate it: data, "
    "preprocessing, model, hyperparameters, evaluation metric, baselines. Mark anything underspecified."),
 "Verbatim check": (
    "Quote verbatim, with citation, every sentence in the sources that supports the claim: '{claim}'. "
    "If none exists, reply exactly: NO DIRECT SUPPORT."),
 "Level-set": (
    "Explain {concept} at three levels: (1) one sentence for a lay reader, (2) a paragraph for a "
    "first-year graduate student, (3) the technical statement with notation as the sources give it."),
 "Draft audit": (
    "My draft is one of the selected sources. List every claim in it that the OTHER sources do not "
    "support, and every claim they contradict. Cite specifically."),
 "Exam generator": (
    "Generate 15 short-answer questions at qualifying-exam difficulty over the selected sources. "
    "Withhold the answers until I respond. Then grade me strictly against the sources."),
 "Timeline": (
    "Construct a chronological timeline of developments in {topic} as documented by the sources, "
    "noting which source establishes each date."),
}

for name, body in PATTERNS.items():
    print(f"\n### {name}\n{body}")

print("\n" + "="*78)
print("Rule of thumb: constrain the OUTPUT SHAPE (table/timeline/levels) and the")
print("EVIDENCE STANDARD ('verbatim', 'not stated', 'NO DIRECT SUPPORT'). Vague")
print("prompts get you fluent prose; constrained prompts get you checkable claims.")


### Evidence table
Build a markdown table over ALL selected sources with columns: source | year | research question | method | sample | key finding | stated limitation. One row per source. Write 'not stated' where the source is silent -- do not infer.

### Disagreement map
Identify every point on which my sources disagree about {topic}. For each: state the positions, name the sources holding each, and quote the sentence that establishes the position.

### Gap finding
What questions about {topic} do these sources raise but NOT answer? Distinguish (a) explicitly flagged as future work from (b) gaps you infer. Label each.

### Steelman / critique
Argue AGAINST the main claim of {source}, using only evidence from the other selected sources. If the other sources contain no such evidence, say so explicitly.

### Methods extraction
Describe the methodology of {source} in enough detail that I could replicate it: data, preprocessing, model, hyperparameters, evaluation metric, baselines. Mark a

---
## 15. Build the mechanism yourself

To *understand* source grounding, implement the smallest honest version: TF-IDF retrieval + extractive answering with citations. No dependencies, no model — which is exactly the point: **the grounding discipline is architectural, not a property of the LLM.** Swapping in Gemini for the final step changes fluency, not provenance.

In [ ]:
# --- A source-grounded QA system in ~45 lines. Pure Python, no dependencies. ---
import math, re
from collections import Counter

# 1. The "corpus" -- two fictional sources -----------------------------------
SOURCES = {
    "smith2024.pdf": """Retrieval-augmented generation reduces hallucination by conditioning
        the decoder on retrieved passages. In our evaluation on 500 questions, RAG lowered
        unsupported-claim rate from 38 percent to 12 percent. However, retrieval failure
        remained the dominant residual error, accounting for 61 percent of remaining mistakes.""",
    "jones2025.pdf": """We argue that citation interfaces change user behaviour more than they
        change model accuracy. Participants who saw inline citations verified claims 3.4 times
        more often. Notably, accuracy gains disappeared when the corpus itself contained errors,
        a condition we term provenance laundering.""",
}

STOP = set("""a an the of in on at to for by is are was were be been we our you i it its this
    that and or but if then than as with from more most much how what which who whom when where
    does do did done can could will would should may might not no""".split())

def tok(s):
    return [w for w in re.findall(r"[a-z]+", s.lower()) if w not in STOP and len(w) > 2]

# 2. Chunk ------------------------------------------------------------------
def chunk(text, size=22):
    w = text.split()
    return [" ".join(w[i:i+size]) for i in range(0, len(w), size)]

CHUNKS = [(src, i, c) for src, t in SOURCES.items() for i, c in enumerate(chunk(t))]

# 3. Retriever: TF-IDF + cosine ---------------------------------------------
N   = len(CHUNKS)
DF  = Counter(w for _, _, c in CHUNKS for w in set(tok(c)))
idf = {w: math.log(N / df) + 0.5 for w, df in DF.items()}
OOV = math.log(N) + 0.5            # unseen query terms: heavy, but match nothing
                                   # -> they inflate the query norm and DEPRESS similarity.
def vec(text, query=False):
    tf = Counter(tok(text))
    v  = {w: (1 + math.log(n)) * idf.get(w, OOV if query else 0.0) for w, n in tf.items()}
    nrm = math.sqrt(sum(x * x for x in v.values())) or 1.0
    return {w: x / nrm for w, x in v.items()}

def cos(a, b):
    return sum(a[w] * b.get(w, 0.0) for w in a)

INDEX = [(src, i, c, vec(c)) for src, i, c in CHUNKS]

def retrieve(q, k=2, floor=0.05):
    qv = vec(q, query=True)
    scored = sorted(((cos(qv, v), s, i, c) for s, i, c, v in INDEX), reverse=True)
    return [x for x in scored[:k] if x[0] >= floor]      # <-- floor == the refusal threshold

# 4. Grounded "generation": extractive, and ALWAYS cited ---------------------
def answer(q, k=2, floor=0.05):
    print(f"Q: {q}\n")
    hits = retrieve(q, k, floor)
    if not hits:
        print("A: The sources do not discuss this.   <-- correct closed-world behaviour\n")
        return
    for score, src, i, c in hits:
        print(f"A: ...{' '.join(c.split())}...")
        print(f"   [{src} - chunk {i} - relevance {score:.3f}]\n")

for q in ["How much does RAG reduce hallucination?",
          "What is provenance laundering?",
          "Do citations change user behaviour?",
          "What is the capital of France?"]:      # <-- outside the corpus
    answer(q)
    print("-" * 78)

In [ ]:
# The refusal threshold is a TUNABLE, not a truth. Sweep it against two queries:
#   Q1 IS answered by the corpus.   Q2 is NOT (nothing here discusses chunking).
Q1 = "How much does RAG reduce hallucination?"
Q2 = "How does chunking affect accuracy?"

print(f"{'floor':>7} | {'Q1 (in corpus)':<34} | {'Q2 (NOT in corpus)':<34}")
print("-" * 82)
for floor in (0.35, 0.10, 0.05):
    row = []
    for q, in_corpus in ((Q1, True), (Q2, False)):
        hits = retrieve(q, k=1, floor=floor)
        if not hits:
            row.append("refuses  " + ("<- F1 MISS" if in_corpus else "correct"))
        else:
            s, src, _, _ = hits[0]
            row.append(f"{src} @{s:.3f} " + ("correct" if in_corpus else "<- F4 SPURIOUS"))
    print(f"{floor:>7.2f} | {row[0]:<34} | {row[1]:<34}")

print("""
Read the table, not the code:

  floor 0.35 -> refuses a question the corpus ANSWERS          (F1: retrieval miss)
  floor 0.10 -> both correct                                   (the sweet spot...)
  floor 0.05 -> grounds an unanswerable question in an
                irrelevant passage, WITH a citation            (F4: provenance laundering)

The usable window is narrow, and it moves with the query and the corpus. Every RAG
system -- Google's included -- picks a point on this curve, and does not disclose it
to you. An answer that arrives with a citation tells you the retriever cleared some
threshold; it does not tell you the passage supports the claim.

That is the whole argument for section 11: click the citation.""")

**What the toy demonstrates, and what it omits**

| Present in the toy | What the real system adds |
|---|---|
| Chunking | Long-context ingestion — often *no* chunking needed |
| Lexical TF-IDF retrieval | Dense embeddings + hybrid + reranking (catches paraphrase) |
| A relevance floor → refusal | Learned/instructed refusal, better calibrated |
| Extractive, citation-attached answers | Abstractive synthesis by Gemini, citations still attached |
| — | Multimodal ingest (audio/video/image), Studio renderings, code execution |

**The load-bearing point:** nothing in the toy required a language model. Chunking, retrieval,
a refusal threshold, and citation attachment are *architectural*. Swapping in Gemini for the final
step buys you fluent, abstractive synthesis — it does not buy you provenance. Provenance came from
the architecture, and so did every failure mode in §10.

---
## 16. Glossary

| Term | Definition |
|---|---|
| **Source grounding** | Constraining generation to a user-supplied corpus; the product's core contract |
| **RAG** | Retrieval-Augmented Generation: retrieve relevant passages, condition generation on them |
| **Closed-domain / closed-world** | Answers only from $\mathcal{D}$; out-of-corpus questions get refusals |
| **Chunking** | Splitting documents into retrievable passages |
| **Context window** | Tokens the model can attend to at once (~$10^6$ for Gemini-class) |
| **Citation chip** | Inline UI element linking a claim to its exact source passage |
| **Studio** | The artifact-generation panel (audio, video, quiz, mind map, …) |
| **Audio / Video Overview** | Podcast- and video-format renderings; audio has an interactive mode |
| **Deep Research** | Agentic multi-site web research producing a cited report |
| **Notebook** | One bounded corpus + its chat + its artifacts; the unit of isolation |
| **Provenance laundering** | A weak source acquiring apparent authority by being cited like a strong one |
| **Retrieval miss (F1)** | Relevant passage exists but is not retrieved — the dominant modern error |
| **Hallucination** | Assertion unsupported by the corpus (grounding reduces, does not eliminate) |

---
## 17. Self-check

Answer without scrolling up.

1. Why is a citation from Gemini Notebook epistemically different from a citation produced by a general chatbot?
2. Your notebook has 250 sources and answers are getting vague. Give two mechanisms and one fix each.
3. You uploaded a PDF and the model insists it doesn't discuss your topic. Diagnose in order of prior probability.
4. Why does the 2026 code-execution feature address a failure grounding *cannot* address?
5. Which Studio artifact has the strongest learning-science support, and why is it not the podcast?
6. You are handling unpublished interview transcripts. Which account, and what do you check first?
7. Give one question type where Notebook beats a full-web chatbot, and one where it is strictly worse.
8. In the §15 toy, what happens to F1 vs F3 errors as you lower `floor`? What's the real-world analogue?

<details><summary><b>Answer sketches</b></summary>

1. The claim was *generated conditioned on* that passage, and the corpus is closed — so the citation is a causal record, not a post-hoc plausible-looking reference.
2. (a) Retrieval dilution → deselect sources / split the notebook; (b) corpus exceeds effective context → curate to a sharper question.
3. F5 scanned/no text layer → F1 vocabulary mismatch → wrong sources selected → the doc genuinely doesn't say it.
4. Grounding fixes provenance, not computation: a correctly-cited but mis-summed number is still wrong. Execution makes numeric claims deterministic.
5. Quiz (retrieval practice / testing effect). Podcasts drive recognition and fluency-illusion, not recall.
6. University **Workspace** account (stronger data terms, no human review); first check your institution's approved-processor / IRB list.
7. Better: "what do my sources *not* say about X." Worse: anything requiring post-upload or out-of-corpus knowledge.
8. Lower floor → fewer F1 refusals, more F3 spurious groundings. Real analogue: every RAG system tuning recall against precision; hence mandatory citation-clicking.

</details>

---
## 18. Sources & further reading

- [NotebookLM is now Gemini Notebook — Google Blog (2026-07-16)](https://blog.google/innovation-and-ai/products/gemini-notebook/notebooklm-gemini-notebook/)
- [Google continues its renaming streak — TechCrunch (2026-07-16)](https://techcrunch.com/2026/07/16/google-continues-its-renaming-streak-by-turning-notebooklm-to-gemini-notebook/)
- [Add or discover new sources — Gemini Notebook Help](https://support.google.com/gemininotebook/answer/16215270)
- [Privacy and Terms of Use in Gemini Notebook — Google Support](https://support.google.com/notebooklm/answer/17004255)
- [Audio & Video Overviews, more languages, longer content — Google Blog](https://blog.google/innovation-and-ai/models-and-research/google-labs/notebook-lm-audio-video-overviews-more-languages-longer-content/)
- [Video Overviews and an upgraded Studio — Google Blog](https://blog.google/innovation-and-ai/models-and-research/google-labs/notebooklm-video-overviews-studio-upgrades/)
- [*NotebookLM: An LLM with RAG for active learning and collaborative tutoring* — arXiv:2504.09720](https://arxiv.org/html/2504.09720v2)
- [What Is NotebookLM? Features and How to Use It in 2026 — DigitalOcean](https://www.digitalocean.com/resources/articles/what-is-notebooklm)
- [NotebookLM: Document-Grounded AI by Google — Emergent Mind](https://www.emergentmind.com/topics/notebooklm)
- [Turn NotebookLM on or off for users — Google Workspace Admin Help](https://knowledge.workspace.google.com/admin/users/access/turn-notebooklm-on-or-off-for-users)
- [Gemini Notebook plans & pricing (official)](https://notebooklm.google/plans)
- [A guide to NotebookLM data security — Devoteam](https://www.devoteam.com/expert-view/a-guide-to-notebooklm-data-security/)

> **Caveat on numbers:** tier limits and feature availability in §8 change on a scale of weeks and are partly sourced from third-party trackers. Verify against Google's own pages before citing them anywhere that matters.